# Milestone 2 – shared setup test

**Team TCM Dynamics · Member C (Bhuwaneshwaree Ragavan)**

This notebook tests the two shared files that every Milestone 2 experiment uses:

- `src/heart_data.py` – loads the fixed HeaRT split, the random ("easy") negatives and the HeaRT hard negatives
- `src/evaluate.py` – scores any model in both settings with the same metrics

It runs Common Neighbours, Adamic–Adar and a default GAE on Cora as a smoke test. These are **not** tuned results.

## 1. Setup

In [ ]:
!pip -q install torch_geometric

import os, sys
REPO = "gnn-link-prediction"
if not os.path.exists(REPO):
    !git clone -q https://github.com/dhanishta-r/gnn-link-prediction.git
SRC = os.path.join(REPO, "src")

# If the shared files are not on GitHub yet, upload them by hand (Colab only).
if not os.path.exists(os.path.join(SRC, "heart_data.py")):
    from google.colab import files
    os.makedirs(SRC, exist_ok=True)
    for name, content in files.upload().items():     # choose heart_data.py and evaluate.py
        open(os.path.join(SRC, name), "wb").write(content)

sys.path.insert(0, SRC)
ROOT = "data"                                        # the HeaRT archive is unpacked here
RESULTS = os.path.join(REPO, "results", "m2", "smoke_test.csv")

## 2. Load the data and check for leaks

The first call downloads the HeaRT archive. Every line under "must be 0" has to be 0. The "allowed" lines count hard negatives that are held-out true edges: HeaRT only filters the edges known at that stage, so we keep them as shipped.

In [ ]:
from heart_data import load_heart, check_no_leak

datasets = {}
for name in ["cora", "citeseer", "pubmed"]:
    datasets[name] = load_heart(name, root=ROOT)
    print(datasets[name].summary())
    check_no_leak(datasets[name])
    print()

## 3. Heuristics on Cora

A model is anything that maps node pairs to scores. The evaluator calls it on the positives, the easy negatives and the hard negatives.

In [ ]:
import numpy as np
from evaluate import evaluate_model, append_results, format_rows

data = datasets["cora"]
A = data.adj                                         # training edges only

def common_neighbours(edges):
    return np.asarray(A[edges[:, 0]].multiply(A[edges[:, 1]]).sum(axis=1)).ravel()

deg = np.asarray(A.sum(axis=0)).ravel()
w = np.zeros_like(deg); w[deg > 1] = 1.0 / np.log(deg[deg > 1])
A_aa = A.multiply(w).tocsr()                         # weight each common neighbour by 1 / log(degree)

def adamic_adar(edges):
    return np.asarray(A[edges[:, 0]].multiply(A_aa[edges[:, 1]]).sum(axis=1)).ravel()

if os.path.exists(RESULTS):
    os.remove(RESULTS)                               # start the smoke-test file fresh
for model_name, fn in [("CN", common_neighbours), ("AA", adamic_adar)]:
    rows = evaluate_model(fn, data, split="test")
    print(model_name); print(format_rows(rows)); print()
    append_results(RESULTS, rows, dataset=data.name, model=model_name, seed=0, member="C", notes="smoke test")

## 4. Default GAE on Cora

Same encoder as the official `autoencoder.py` (GCN, 32 hidden, 16 output, lr 0.01), trained on the HeaRT training edges only.

In [ ]:
import torch
from torch_geometric.nn import GAE, GCNConv

class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 2 * out_channels)
        self.conv2 = GCNConv(2 * out_channels, out_channels)
    def forward(self, x, edge_index):
        return self.conv2(self.conv1(x, edge_index).relu(), edge_index)

def run_gae(data, seed, epochs=200, lr=0.01, out_channels=16):
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    x, edge_index = data.to_torch()
    x, edge_index = x.to(device), edge_index.to(device)
    model = GAE(GCNEncoder(x.size(1), out_channels)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train(); opt.zero_grad()
        z = model.encode(x, edge_index)
        loss = model.recon_loss(z, edge_index)       # negatives are sampled inside
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        z = model.encode(x, edge_index).cpu()
    def score(edges):
        e = torch.as_tensor(edges)
        return (z[e[:, 0]] * z[e[:, 1]]).sum(-1)
    return score

for seed in range(2):
    score = run_gae(data, seed)
    for split in ["valid", "test"]:
        rows = evaluate_model(score, data, split=split)
        print(f"GAE seed {seed}"); print(format_rows(rows)); print()
        append_results(RESULTS, rows, dataset=data.name, model="GAE", config="default,epochs=200",
                       seed=seed, member="C", notes="smoke test")

## 5. The shared results file

Every experiment appends rows in this layout. `setting` is `easy` or `heart`; `split` is `valid` (tuning) or `test` (final). All metrics are in percent.

In [ ]:
import pandas as pd
pd.read_csv(RESULTS)

## 6. How to use this in your own experiments

```python
from heart_data import load_heart, check_no_leak
from evaluate import evaluate_model, append_results

data = load_heart("cora")              # or "citeseer", "pubmed"
x, edge_index = data.to_torch()        # training graph for PyTorch Geometric
# ... train your model on data.train_pos only ...

def score_fn(edges):                   # (N, 2) numpy array -> N scores
    ...

rows = evaluate_model(score_fn, data, split="valid")    # use for tuning
rows = evaluate_model(score_fn, data, split="test")     # only for the chosen configuration
append_results("results/m2/raw_seal.csv", rows, dataset=data.name,       # your own file, see below
               model="SEAL", config="hops=2,hidden=32", seed=0, member="A")
```

Rules we agreed on:

1. Train only on `data.train_pos`. Never add validation or test edges to the graph.
2. Choose hyper-parameters on `split="valid"`. Run `split="test"` only for the final configuration.
3. Same tuning budget for every model, then 10 seeds (0–9) of the chosen configuration.
4. Write every run, including the unsuccessful ones, to your own file: `results/m2/raw_seal.csv` (SEAL), `results/m2/raw_tuning.csv` (tuning) or `results/m2/raw_baselines.csv` (baselines). `src/aggregate_m2.py` merges them.

The full plan is in `docs/milestone2_handover.md`.